# Individual Energy-Profile Prediction — realism evaluation

The per-activity **individual curve** prediction benchmark (Baseline, ML DTW,
ML + Ext. Factors, Seq2Seq, …), scored on **curve realism only**.

The pointwise metrics (sMAE, sRMSE, WAPE) are deliberately **not** reported here.
They cannot separate a usable load profile from a flat line drawn through the
middle of one — they *prefer* the flat line — so every number below compares a
**property** of the predicted curve with the same property of the real one.

- **Anonymised**: process, activity and sensor names are replaced by shuffled
  numbered labels (`Process 4`, `Activity 17`, `Sensor 23`). Only the approach
  names and the metrics are readable; the label numbers carry no ordering.
- Source: `curve_eval_results.parquet` (per-curve records with an **Activity**
  dimension).
- Granularity: aggregated **process × activity × sensor**, then across those with
  the **median**. The median is the only aggregation reported.
- One **aggregated** table (all processes) as a styled table and as LaTeX.
- **Boxplots** of the aggregated per-(process, activity, sensor) values.
- A closing **breakdown**, in tables only: per unit, per activity and per sensor,
  each with the number of curves and cells behind it.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

EXPERIMENT = 1000
SPLIT      = 'TEST'                       # 'TEST' or 'TRAIN'
BOX_METRIC = 'Std'                        # realism metric that orders boxplots
                                          # and the per-process tables
EXCLUDE_APPROACHES = ['Exemplar (real curve)', 'Exemplar (no DTW)',
                      'Exemplar + DTW (warped)']   # raw names; dropped everywhere below
SAVE_LATEX = True

# ── Anonymisation ────────────────────────────────────────────────────────────
# Process, Activity and Sensor names are replaced by numbered labels everywhere
# they could surface — tables, prints, LaTeX — so nothing below identifies a
# plant, a step or an instrument. Only the APPROACH names and the metrics stay
# readable, which is all the results need.
# The numbering is SHUFFLED against the sorted real names, so 'Process 1' is not
# the first process, 'Activity 1' not the first activity, and the label order
# carries no information about the underlying order. ANON_SEED fixes the
# permutation, so a given seed always produces the same labels and a table can be
# regenerated to match one already in the paper; change it to re-draw them.
ANONYMISE = True         # <- the switch. The button below just sets this.
ANON_SEED = 20260730

# The button. ipywidgets only renders in a LIVE kernel, and clicking it cannot
# retroactively change cells that have already run — it sets ANONYMISE and you
# re-run from the load cell down. Editing the line above does exactly the same
# thing, which is why the flag, not the widget, is the source of truth: a notebook
# opened without ipywidgets (or exported to HTML/PDF) still behaves correctly.
try:
    import ipywidgets as _w
    _anon_note = _w.HTML()
    _anon_btn = _w.ToggleButtons(
        options=[('🔒 Anonymised labels', True), ('👁 Real names', False)],
        value=ANONYMISE, description='Labels:',
        style={'button_width': '190px', 'description_width': '60px'})

    def _on_anon(change):
        global ANONYMISE
        ANONYMISE = change['new']
        _anon_note.value = (
            '<span style="color:#b35c00">Now <b>{}</b> — re-run the load cell and '
            'everything below for it to take effect.</span>'.format(
                'ANONYMISED' if ANONYMISE else 'REAL NAMES'))

    _anon_btn.observe(_on_anon, names='value')
    display(_w.VBox([_anon_btn, _anon_note]))
except ImportError:
    print(f'ipywidgets not installed — set ANONYMISE by hand (currently {ANONYMISE}).')

# ── Realism metrics — the ONLY metrics this notebook reports ─────────────────
# The pointwise metrics (sMAE, sRMSE, WAPE) are computed by the pipeline and
# deliberately dropped here. Pointwise error cannot tell a usable load profile
# from a flat line drawn through the middle of one; it actively prefers the flat
# line, because sMAE z-scores by the true curve and takes a mean absolute
# deviation, so a smooth prediction through a spiky signal scores well. That is
# how the averaging approaches win it while emitting roughly a tenth of the real
# variation — which is exactly what this notebook is here to measure instead.
#
# Identical logic to 05_results_complete_energy_profile.ipynb: each metric is the
# NORMALISED ABSOLUTE ERROR of a property of the predicted curve against the same
# property of the real one,
#
#     err = |f_pred - f_real| / scale,   scale = mean|f_real| over that
#                                                (process, sensor)'s real curves
#
# so 0 = perfect and LOWER IS BETTER, exactly like sMAE/WAPE and exactly like the
# complete-profile notebook. Same names, same formula, same direction, so 'Std'
# means the same number in both notebooks. 'Overall' (added to the tables below)
# is the plain average across the metric columns, as it is there.
#
# Why normalised rather than the ratio f_pred/f_real this notebook used before:
#   * the ratio's denominator is THAT CURVE'S OWN f_real, so a near-flat real
#     curve (std_real ~ 0) explodes or has to be dropped. Dividing by the
#     sensor's TYPICAL real magnitude is robust to individual degenerate curves.
#   * the ratio is asymmetric: 0.5 and 2.0 are both "factor 2 wrong" but sit 0.5
#     and 1.0 from 1.0, so a median of ratios UNDER-penalises flatness — exactly
#     the failure these metrics exist to expose.
# Both are unitless, so either supports the cross-sensor aggregate table that raw
# values could never support (degC, bar and kW in one column).
#
# The cost of the absolute value is DIRECTION: 'Std' 0.20 says the spread is off
# by a fifth of the sensor's typical spread, not whether the curve is too flat or
# too excited. Every approach in this data is too flat; add a signed companion
# column if that stops being true.
#
# 'Sum' is derived as mean x N in the load cell — see there. AC1 and the
# duty-cycle/zero-fraction metric are deliberately absent; duty cycle is measured
# properly, relative to each curve's own range, in the complete-profile notebook.
#
#   name -> (predicted column, real column, target)
REALISM_SPECS = {
    'Sum':       ('sum_pred',   'sum_real',   0.0),   # total energy over the curve (= mean x N)
    'Max':       ('max_pred',   'max_real',   0.0),   # peak, which sizes equipment and tariffs
    'Mean':      ('mean_pred',  'mean_real',  0.0),   # average load level, i.e. the bias
    'Std':       ('std_pred',   'std_real',   0.0),   # spread — THE flatness measure
    'Roughness': ('rough_pred', 'rough_real', 0.0),   # jaggedness: catches the OPPOSITE failure to
                                                      # 'Std' -- right amplitude reached by adding
                                                      # noise rather than real dynamics
}
REALISM        = list(REALISM_SPECS)
REALISM_TARGET = {k: v[2] for k, v in REALISM_SPECS.items()}

# These need the per-curve shape statistics sim_extractor._curve_shape_stats
# writes into curve_eval_results.parquet. Runs produced BEFORE that existed
# (experiment_969 and earlier) stored only the five pointwise metrics, so every
# realism section below detects the missing columns and skips itself with a note
# instead of failing. Re-run the pipeline to populate them.

# ── Levelling: two independent ways the approaches are scored on different
#    populations. Both are on by default; turn either off to see the raw
#    behaviour. Section 0 verifies the result and complains if anything is left.

# 1) Curve-length floor, applied to EVERY approach before any table or figure.
# The pipeline does not score the same curves for every approach: the
# *_external ones are evaluated through split_curves_with_prev_activity, which
# keeps curves with >= 5 samples, while every other approach goes through
# split_curves, which keeps >= 2. Curves 2-4 samples long therefore exist for
# six approaches and not for the two prev-activity ones (experiment_964:
# 30,080 vs 26,000 TEST curves), so the raw columns are medians over different
# populations. Filtering here levels them; set to 0 to see the raw behaviour.
MIN_CURVE_POINTS = 5

# 2) Cell coverage. MIN_CURVE_POINTS equalises curve *length*, not WHICH
# (process, activity, sensor) cells an approach is scored on — and the Baseline
# is not scored on the same ones. It is the median curve of a sensor, so it
# exists for every activity, while a learned approach only has a pipeline where
# there was enough training data to fit one. In experiment_968 that left the
# Baseline holding 442 cells / 19,739 TEST curves against 390 / 19,681 for the
# other seven, and its 52 extra cells are the hard ones (median sMAE 3.5, max
# 63.9): the Baseline row was being penalised for activities no other method was
# ever asked to predict. Restricting to the cells every approach owns makes the
# populations identical — same curves, same cells, same non-NaN cells per metric.
LEVEL_CELL_COVERAGE = True

# ── Method names as they should read everywhere (tables, LaTeX, boxplots) ────
# Keys are the raw `Approach` values in curve_eval_results.parquet; anything not
# listed keeps its raw name. Applied right after loading, so every downstream
# table and figure uses these names. NOTE: the raw 'Median per Activity & Sensor'
# must not survive into LaTeX — a bare '&' would be read as a column separator.
METHOD_RENAME = {
    # Paper naming scheme: the proposed method is 'Step DTW'; every ablation is
    # named by what was removed; the two medians are named by their granularity.
    'Step DTW smooth + ML + Ext.':             'Step DTW (ours)',
    'Step DTW + ML + Ext.':                    'Step DTW (step gains)',
    'ML + Ext. Factors':                       'Step DTW w/o segments',
    'DTW + ML + Ext. Factors':                 'Step DTW w/o segments',
    'ML DTW':                                  'Step DTW w/o segments and ext.',
    'DTW + ML':                                'Step DTW w/o segments and ext.',
    'ML only (no DTW)':                        'Step DTW w/o DTW',
    'Median per Activity & Sensor':            'Activity median',
    'Median per activity and sensor':          'Activity median',
    'Baseline':                                'Sensor median (baseline)',
    'DTW + Seq2Seq + Ext. Factors':            'Seq2Seq (aligned)',
    'DTW + Seq2Seq':                           'Seq2Seq (aligned, w/o ext.)',
    'Seq2Seq only (no DTW)':                   'Seq2Seq (unaligned)',
    'Exemplar (real curve)':                   'Exemplar (real curve)',
}

results_root = Path('..') / 'results'
runs = sorted([d for d in results_root.iterdir()
               if d.is_dir() and d.name.startswith(f'experiment_{EXPERIMENT}_')])
assert runs, f'No runs for experiment {EXPERIMENT}'
run_dir = runs[-1]
print('Using run:', run_dir.name)

Using run: experiment_1000_20260729_201444


In [2]:
# ── Load + aggregate to (process, activity, sensor, approach) ────────────────
raw = pd.read_parquet(run_dir / 'curve_eval_results.parquet')
raw = raw[raw['Split'] == SPLIT]
if EXCLUDE_APPROACHES:                       # exclusion uses the RAW names
    raw = raw[~raw['Approach'].isin(EXCLUDE_APPROACHES)]

# ── Real test curves — stored once, joined back here ─────────────────────────
# y_pred lives on every scored row (it differs per approach); y_true does NOT,
# because every approach is scored on the SAME curves and repeating the array
# once per approach was ~half the file duplicated tenfold. 02_modelling.py writes it
# to real_test_curves.parquet keyed on the curve identity; this merge restores
# the pairing so `raw` has both arrays exactly as before.
_real_path = run_dir / 'real_test_curves.parquet'
if 'y_pred' in raw.columns and 'y_true' not in raw.columns and _real_path.exists():
    _real = pd.read_parquet(_real_path)
    _rc_key = [c for c in ['Process', 'Sensor', 'Activity', 'Instance', 'Split']
               if c in raw.columns and c in _real.columns]
    raw = raw.merge(_real[_rc_key + ['y_real']], on=_rc_key, how='left')
    raw = raw.rename(columns={'y_real': 'y_true'})
    print(f'Joined {len(_real):,} real curves from real_test_curves.parquet '
          f'on {" + ".join(_rc_key)}  '
          f'({raw["y_true"].notna().mean()*100:.1f}% of scored rows matched)')
elif 'y_pred' in raw.columns and 'y_true' not in raw.columns:
    print('NOTE: y_pred is present but real_test_curves.parquet is not — '
          'per-curve arrays are only half available for this run.')

# ── Anonymise the identity columns (see ANONYMISE in the config cell) ────────
# Applied HERE, after the real-curve merge and before anything reads a label:
# the merge above joins raw to real_test_curves.parquet on the real names, so
# renaming any earlier would break the join, and every table, print and LaTeX
# string below is produced after this point.
def _anon_map(values, prefix, seed):
    """Sorted real labels -> '<prefix> <n>', with n a random permutation of 1..N."""
    labels = sorted(pd.Series(values).dropna().astype(str).unique())
    order  = np.random.default_rng(seed).permutation(len(labels)) + 1
    return {lab: f'{prefix} {n}' for lab, n in zip(labels, order)}

ANON_MAPS = {}
if ANONYMISE:
    # A different seed per column, else columns with the same number of labels
    # would receive the same permutation and the two could be lined up.
    for _i, _col in enumerate(['Process', 'Activity', 'Sensor']):
        if _col in raw.columns:
            ANON_MAPS[_col] = _anon_map(raw[_col], _col, ANON_SEED + _i)
            raw[_col] = raw[_col].astype(str).map(ANON_MAPS[_col])
    print('Anonymised: ' + ', '.join(f'{c} ({len(m)} labels)' for c, m in ANON_MAPS.items())
          + f'  [seed {ANON_SEED}; ANON_MAPS holds the mapping in memory only]')

CELL_KEY = ['Process', 'Activity', 'Sensor']
_before = raw.groupby('Approach').size()

# ── Realism metrics, derived per curve ───────────────────────────────────────
# Built here so they flow through the levelling and both aggregation stages
# exactly like the pointwise metrics — same curves, same cells, same rules.
# HAVE_REALISM gates every realism section; on an older parquet the shape
# statistics are simply absent and those sections say so instead of failing.
# Per-metric availability, not all-or-nothing: a parquet written before one of
# these shape statistics existed still supports every OTHER realism metric, and
# an all-or-nothing gate would silently throw them all away. rough_* was added
# after experiment_973, so that run keeps std/peak/level/acf1 and loses only the
# roughness column.
# 'Sum' is not written by sim_extractor._curve_shape_stats, but it is EXACTLY
# recoverable: mean = sum/N and the prediction is always emitted at the real
# curve's own length, so N is shared and sum = mean x N on both sides. Derived
# here rather than added to the pipeline so existing parquets support it too.
for _side in ('real', 'pred'):
    if f'mean_{_side}' in raw.columns and 'N' in raw.columns:
        raw[f'sum_{_side}'] = (pd.to_numeric(raw[f'mean_{_side}'], errors='coerce')
                               * pd.to_numeric(raw['N'], errors='coerce'))

_needed  = {c for spec in REALISM_SPECS.values() for c in spec[:2]}
_SPECS   = {k: v for k, v in REALISM_SPECS.items()
            if v[0] in raw.columns and v[1] in raw.columns}
_missing = sorted(_needed - set(raw.columns))
HAVE_REALISM = bool(_SPECS)

if HAVE_REALISM:
    # Scale per (process, sensor): the TYPICAL magnitude of that sensor's real
    # curves for this feature. Deduplicated on Instance first — the same real
    # curve appears once per approach, and without the dedup a sensor scored by
    # more approaches would weigh more heavily in its own scale.
    for _name, (_num, _den, _tgt) in _SPECS.items():
        p = pd.to_numeric(raw[_num], errors='coerce')
        r = pd.to_numeric(raw[_den], errors='coerce')
        _uniq = raw.drop_duplicates(['Process', 'Sensor', 'Instance'])
        _scale_by_cell = (pd.to_numeric(_uniq[_den], errors='coerce').abs()
                          .groupby([_uniq['Process'], _uniq['Sensor']]).mean())
        _scale = pd.Series(
            pd.MultiIndex.from_arrays([raw['Process'], raw['Sensor']])
              .map(_scale_by_cell).to_numpy(dtype=float), index=raw.index)
        # Degenerate-scale guard, same as the complete-profile notebook: if a
        # sensor's real curves have essentially no spread/peak/level at all there
        # is nothing to reproduce, and dividing by it would emit garbage.
        raw[_name] = np.where(np.isfinite(_scale) & (_scale > 1e-6),
                              (p - r).abs() / _scale, np.nan)
    ACTIVE_REALISM = [m for m in REALISM if m in _SPECS and raw[m].notna().any()]
    print(f'Realism metrics available: {ACTIVE_REALISM}')
    if _missing:
        print(f'  (columns absent from this run, metrics skipped: {_missing} — '
              f're-run the pipeline to populate them)')
    _empty = [m for m in _SPECS if m not in ACTIVE_REALISM]
    if _empty:
        print(f'  (no finite values for {_empty} — denominator is zero everywhere, '
              f'e.g. a sensor whose real curves have no spread at all)')
else:
    ACTIVE_REALISM = []
    print('⚠️ No realism metrics in this run: curve_eval_results.parquet has none of '
          f'{sorted(_needed)}.\n'
          '   Those per-curve shape statistics were added to '
          'sim_extractor._curve_shape_stats after this\n'
          '   run was produced, so only the pointwise metrics exist here, and this '
          'notebook reports none of those.\n'
          '   Re-run the pipeline to populate them; every section below will skip '
          'itself until then.')

ALL_METRICS = list(ACTIVE_REALISM)   # realism only — no pointwise columns

# Levelling 1 — curve length (see MIN_CURVE_POINTS).
if MIN_CURVE_POINTS:
    raw = raw[raw['N'] >= MIN_CURVE_POINTS]

# Levelling 2 — cell coverage (see LEVEL_CELL_COVERAGE). Applied AFTER the
# length floor, because dropping short curves can itself empty a cell for one
# approach and not another; intersecting on what survives is what makes the
# populations equal.
if LEVEL_CELL_COVERAGE:
    _cells_of = {a: set(map(tuple, g[CELL_KEY].drop_duplicates().to_numpy()))
                 for a, g in raw.groupby('Approach')}
    _shared = set.intersection(*_cells_of.values()) if _cells_of else set()
    raw = raw[raw[CELL_KEY].apply(tuple, axis=1).isin(_shared)]
    _own_only = {a: len(c - _shared) for a, c in _cells_of.items() if c - _shared}
    print(f'Cell coverage levelled to the {len(_shared)} (process, activity, sensor) '
          f'cells every approach is scored on')
    for _a, _n in sorted(_own_only.items()):
        print(f'  {_a}: dropped {_n} cells no other approach has')

_after = raw.groupby('Approach').size()
_levelling = pd.DataFrame({'curves_raw': _before, 'curves_used': _after}).fillna(0).astype(int)
_levelling['dropped'] = _levelling['curves_raw'] - _levelling['curves_used']
display(Markdown(
    f'**Population levelled** — curves with $\\geq$ {MIN_CURVE_POINTS} samples'
    + (', on the (process, activity, sensor) cells shared by every approach'
       if LEVEL_CELL_COVERAGE else '')))
display(_levelling)
if _levelling['curves_used'].nunique() > 1:
    print('⚠️ curve counts still differ across approaches after levelling — '
          'the comparison is not like-for-like.')

raw['Approach'] = raw['Approach'].replace(METHOD_RENAME)   # display names from here on
print(f'{len(raw):,} per-curve rows | approaches: {raw["Approach"].nunique()} | '
      f'processes: {sorted(raw["Process"].unique())}')
print('methods:', sorted(raw['Approach'].unique()))

# Stage 1: one value per (process, activity, sensor, approach) = median over its
# curves. Median at BOTH stages — these errors are bounded below with a long right
# tail, so a handful of badly-predicted cells would drag any mean, and a mean of
# medians is neither statistic. `combo` is the single frame everything below uses.
_CELL_GROUP = ['Process', 'Activity', 'Sensor', 'Approach']
combo = raw.groupby(_CELL_GROUP)[ALL_METRICS].median().reset_index()

print('per (process, activity, sensor, approach) rows:', len(combo))

Joined 58,478 real curves from real_test_curves.parquet on Process + Sensor + Activity + Instance + Split  (100.0% of scored rows matched)
Realism metrics available: ['Sum', 'Max', 'Mean', 'Std', 'Roughness']
Cell coverage levelled to the 390 (process, activity, sensor) cells every approach is scored on
  Baseline: dropped 52 cells no other approach has


**Population levelled** — curves with $\geq$ 5 samples, on the (process, activity, sensor) cells shared by every approach

,curves_raw,curves_used,dropped
Approach,,,
Baseline,19403,19345,58
ML + Ext. Factors,19345,19345,0
ML only (no DTW),19345,19345,0
Median per Activity & Sensor,19345,19345,0
Step DTW smooth + ML + Ext.,19345,19345,0


96,725 per-curve rows | approaches: 5 | processes: ['process_1', 'process_2', 'process_3', 'process_4_1', 'process_4_2', 'process_5']
methods: ['Baseline', 'DTW + ML + Ext. Factors', 'ML only (no DTW)', 'Median per activity and sensor', 'Step DTW smooth + ML + Ext.']
per (process, activity, sensor, approach) rows: 1950


In [3]:
# ── Table helpers (bold best per column; every realism metric has target 0) ──
# Row labels come from METHOD_RENAME, which is already applied to the data in the
# load cell; it is reapplied here so the tables stay correct even if that cell is
# re-run out of order.
# Column headers in the paper table. Realism names are short enough to use as-is;
# override any that need a different label in LaTeX.
PAPER_HDR = {'Roughness': 'Roughness'}
# Widths of the paper float — tweak if the table doesn't fit the column.
PAPER_MINIPAGE = '8.5cm'    # minipage holding caption + tabular
PAPER_FIRSTCOL = '6cm'      # p{} width of the method column
PAPER_NOTEBOX  = '12cm'     # parbox width of the note under the table
PAPER_DECIMALS = 3

def _tex(s):
    # Escape the LaTeX specials that can appear in method / process names.
    return (str(s).replace('\\', r'\textbackslash ').replace('&', r'\&')
                  .replace('_', r'\_').replace('%', r'\%').replace('#', r'\#'))

def _row_label(idx):
    return _tex(METHOD_RENAME.get(idx, idx))

# ── How far a realism value is from its target ───────────────────────────────
# Every realism metric is a normalised ABSOLUTE error with target 0, so the
# distance to target is just the value. Kept as a function because the tables,
# LaTeX and composite all rank through it, and a signed metric added later must
# not silently make highlight_min bold the worst cell.
def realism_deviation(values, metric):
    v = pd.to_numeric(pd.Series(values), errors='coerce')
    return (v - REALISM_TARGET.get(metric, 0.0)).abs()


# Stage 2: collapse the per-(process, activity, sensor) cells into one row per
# approach. Median at both stages — see the note in the load cell.
def approach_table(df_combo, how='median', metrics=None):
    metrics = list(metrics if metrics is not None else ACTIVE_REALISM)
    t = df_combo.groupby('Approach')[metrics].agg(how)
    sort_by = BOX_METRIC if BOX_METRIC in t.columns else metrics[0]
    # Every column here has a target, not a direction: rank by distance to it.
    return t.reindex(realism_deviation(t[sort_by], sort_by).sort_values().index)

def style_realism(t, how='median'):
    """
    Realism columns are normalised absolute errors with target 0, so lower is
    better and the best cell is the minimum.
    """
    def _best(col):
        d = realism_deviation(col, col.name)
        return ['font-weight:700;background-color:#d6ecff;'
                if (pd.notna(v) and pd.notna(d.min()) and abs(v - d.min()) < 1e-12)
                else '' for v in d]
    targets = ', '.join(f'{m} → {REALISM_TARGET.get(m, 0.0):g}' for m in t.columns)
    return (t.style.format({m: '{:.3f}' for m in t.columns}, na_rep='—')
             .apply(_best, axis=0)
             .set_caption(f'{SPLIT} — {how} over process×activity×sensor; '
                          f'target: {targets}. Distance from the target is what counts, '
                          f'in EITHER direction.'))

def _cells(t, dp=3):
    # Formatted strings with the per-column best bolded — the value closest to
    # that column's target.
    s = pd.DataFrame(index=t.index, columns=t.columns, dtype=object)
    for col in t.columns:
        vals = t[col].dropna()
        _d = realism_deviation(vals, col).dropna()
        best = vals.get(_d.idxmin()) if not _d.empty else None
        for idx in t.index:
            v = t.loc[idx, col]
            s.loc[idx, col] = ('' if pd.isna(v) else
                               (r'\textbf{' + f'{v:.{dp}f}' + '}')
                               if (best is not None and abs(v - best) < 1e-6)
                               else f'{v:.{dp}f}')
    return s

def to_latex(t, caption, label):
    s = _cells(t)
    s.index = [_row_label(i) for i in t.index]
    latex = s.to_latex(escape=False, column_format='l|' + '|'.join(['c']*len(t.columns)),
                       caption=caption, label=label, position='H')
    return latex

def to_latex_paper(t, caption, label, note):
    # Paper float: table* + minipage, caption on top, booktabs rules, bold header
    # row, and a \parbox note underneath. Needs booktabs + caption in the preamble.
    s = _cells(t, PAPER_DECIMALS)
    body = '\n'.join(' & '.join([_row_label(idx)] + [s.loc[idx, c] for c in t.columns]) + r' \\'
                     for idx in t.index)
    hdr = ' & '.join([r'\textbf{Method}']
                     + [r'\textbf{' + PAPER_HDR.get(c, _tex(c)) + '}' for c in t.columns])
    return '\n'.join([
        r'\begin{table*}[H]',
        r'\centering',
        '',
        rf'\begin{{minipage}}{{{PAPER_MINIPAGE}}}',
        r'\centering',
        '',
        r'\captionsetup{',
        r'    justification=centering,',
        r'    singlelinecheck=false,',
        r'    format=plain',
        r'}',
        '',
        rf'\caption{{{caption}}}',
        rf'\label{{{label}}}',
        '',
        r'\vspace{-0.5em}',
        '',
        rf'\begin{{tabular}}{{p{{{PAPER_FIRSTCOL}}}|' + '|'.join(['c'] * len(t.columns)) + '}',
        r'\toprule',
        hdr + r' \\',
        r'\midrule',
        body,
        r'\bottomrule',
        r'\end{tabular}',
        '',
        r'\vspace{0.5em}',
        '',
        rf'\parbox{{{PAPER_NOTEBOX}}}{{%',
        r'\footnotesize',
        note,
        r'}',
        '',
        r'\end{minipage}',
        '',
        r'\end{table*}',
    ])


# ── Realism scorecard ────────────────────────────────────────────────────────
# Mirrors scorecard() in 05_results_complete_energy_profile.ipynb: the metric
# columns plus an 'Overall' that is their plain average, rows sorted best-first
# by it. Same definition there and here so the two tables are read the same way.
SHOW_OVERALL    = True
SORT_BY_OVERALL = True

def realism_scorecard(df_combo, how='median', metrics=None):
    metrics = list(metrics if metrics is not None else ACTIVE_REALISM)
    t = df_combo.groupby('Approach')[metrics].agg(how)
    if SHOW_OVERALL and len(t.columns):
        t['Overall'] = t.mean(axis=1)       # simple average across the metric columns
        if SORT_BY_OVERALL:
            t = t.sort_values('Overall')    # best first
    return t


## 0 · Evaluation counts per approach

How many evaluations each row of the tables and each box of the plots below is built
from. The medians only compare like with like if every approach is scored on the
**same** (process, activity, sensor) cells; any shortfall is flagged.

In [4]:
# ── Evaluations behind every number below ────────────────────────────────────
# Two populations exist here and they are not the same size:
#   curves -- the per-curve rows of curve_eval_results.parquet (stage-1 input)
#   cells  -- the (process, activity, sensor) medians those curves collapse to.
#             ONE CELL IS ONE UNIT of every median in the tables below and one
#             point in every boxplot.
# The tables only compare like with like if every approach owns the same cells,
# so both counts — plus the non-NaN cells per metric, since a NaN shrinks the
# population of that column alone — are reported before any result.
_KEY = ['Process', 'Activity', 'Sensor']

# Cell-by-cell overlap: equal counts are necessary but not sufficient — two
# approaches can hold the same number of cells without holding the same ones,
# so the shared set is intersected explicitly and reported as its own column.
_sets   = {a: set(map(tuple, g[_KEY].drop_duplicates().to_numpy()))
           for a, g in combo.groupby('Approach')}
_common = set.intersection(*_sets.values()) if _sets else set()
_extra  = {a: len(s - _common) for a, s in _sets.items()}

eval_counts = pd.DataFrame({'curves': raw.groupby('Approach').size(),
                            'cells':  combo.groupby('Approach').size()})
for _m in ACTIVE_REALISM:
    eval_counts[f'{_m} cells'] = combo.groupby('Approach')[_m].count()
eval_counts['shared cells']  = pd.Series({a: len(s & _common) for a, s in _sets.items()})
eval_counts['outside shared'] = pd.Series(_extra)   # cells not every approach has
eval_counts = eval_counts.fillna(0).astype(int).sort_index()

display(Markdown('### Evaluation counts per approach — the population of every median below'))
display(eval_counts)

_DIAG   = ['shared cells', 'outside shared']      # diagnostics, not populations
_uneven = [c for c in eval_counts.columns
           if c not in _DIAG and eval_counts[c].nunique() > 1]

if _uneven or any(_extra.values()):
    print('⚠️ approaches are NOT scored on the same population — the medians and '
          'boxplots below are not like-for-like:')
    for c in _uneven:
        hi = eval_counts[c].max()
        print(f'   {c}: max {hi}, others -> ' +
              ', '.join(f'{a}={v}' for a, v in eval_counts[c].items() if v != hi))
    print(f'   (process, activity, sensor) cells shared by every approach: {len(_common)}')
    for a, n in _extra.items():
        if n:
            print(f'   {a}: {n} cells outside that shared set — they enter this '
                  f'approach\'s median and no other')
else:
    print(f'✅ like-for-like: every approach is scored on the same {len(_common)} '
          f'(process, activity, sensor) cells')

### Evaluation counts per approach — the population of every median below

,curves,cells,Sum cells,Max cells,Mean cells,Std cells,Roughness cells,shared cells,outside shared
Approach,,,,,,,,,
Baseline,19345,390,390,390,390,390,390,390,0
DTW + ML + Ext. Factors,19345,390,390,390,390,390,390,390,0
ML only (no DTW),19345,390,390,390,390,390,390,390,0
Median per activity and sensor,19345,390,390,390,390,390,390,390,0
Step DTW smooth + ML + Ext.,19345,390,390,390,390,390,390,390,0


✅ like-for-like: every approach is scored on the same 390 (process, activity, sensor) cells


## 1 · Realism — aggregated, all processes

Does the predicted curve look like a real one? Every metric compares a **property**
of the predicted curve with the same property of the real curve, normalised by that
sensor's typical real magnitude — the same estimator, names and direction as
`05_results_complete_energy_profile.ipynb`:

$$\mathrm{err} = \frac{|f(\text{pred}) - f(\text{real})|}{\overline{|f(\text{real})|}}$$

so **0 = perfect and lower is better**. `Overall` is the plain average across the
metric columns, as it is there.

| metric | per-curve scalar | reads as |
|---|---|---|
| `Sum` | `sum(v)` | total energy drawn over the curve — the **overall consumption** |
| `Max` | `max(v)` | highest instantaneous load — **peak demand / connection sizing** |
| `Mean` | `mean(v)` | average load level over the curve — the plain **bias** |
| `Std` | `std(v)` | how much the load swings — **amplitude of the dynamics**, the flatness measure |
| `Roughness` | `mean\|Δv\|` | mean absolute step between consecutive samples — **jaggedness / switching intensity** |

`Std` and `Roughness` are complementary: a method can reach the right `Std` by adding
high-frequency noise rather than reproducing real dynamics, and only `Roughness`
separates those two.

Aggregation is the **median** at both stages — over the curves of a
(process, activity, sensor) cell, then over those cells. These errors are bounded
below and have a long right tail, so a handful of badly-predicted cells would drag
any mean; the median reports the typical cell.

In [5]:
if not ACTIVE_REALISM:
    display(Markdown('> **Skipped — this run has no realism metrics.** '
                     'See the warning in the load cell: re-run the pipeline to write '
                     'the per-curve shape statistics into `curve_eval_results.parquet`.'))
else:
    realism_median = realism_scorecard(combo, how='median')
    display(Markdown('#### Realism — median over process×activity×sensor (the reported table)'))
    display(style_realism(realism_median))

#### Realism — median over process×activity×sensor (the reported table)

,Sum,Max,Mean,Std,Roughness,Overall
Approach,,,,,,
Step DTW smooth + ML + Ext.,0.018,0.070,0.048,0.467,0.569,0.234
Median per activity and sensor,0.016,0.072,0.043,0.552,0.662,0.269
DTW + ML + Ext. Factors,0.018,0.080,0.050,0.531,0.676,0.271
ML only (no DTW),0.018,0.089,0.054,0.592,0.702,0.291
Baseline,0.018,0.074,0.051,0.613,0.724,0.296


## 2 · LaTeX — the table for the paper

The aggregated realism table in the paper float layout (`table*` + `minipage` +
caption + note). Needs `booktabs`, `caption` and `float` in the preamble. With
`SAVE_LATEX` the same LaTeX is also written to
`visuals/individual_profile_realism.tex`, ready to `\input{}`.

In [6]:
# The aggregated table in the paper layout (table* + minipage + note). Printed, and
# with SAVE_LATEX also written to a .tex file that can be \input{} directly.

AGG_LABEL = f'tab:individual_profile_realism'   # label the paper \ref's

REALISM_NOTE = (
    'Curve realism for individual profile prediction, {split} set. Each column is the '
    'median over (process, activity, sensor) cells of the per-curve normalised error '
    r'$|f(\mathrm{{pred}})-f(\mathrm{{real}})|\,/\,\overline{{|f(\mathrm{{real}})|}}$, '
    'where $f$ is the property named in the header, so \\textbf{{0 is perfect and lower '
    'is better}}. Sum = total consumption, Max = peak demand, Mean = level (bias), '
    'Std = amplitude of the dynamics, Roughness = mean absolute step between samples; '
    'Overall is their plain average.\n'
    r'\textbf{{Bold}} marks the best value per column.' '\n'
    'Std and Roughness are complementary: a method can reach the right Std by adding '
    'high-frequency noise instead of reproducing real dynamics, and only Roughness '
    'separates those two. Pointwise error metrics are not reported: they cannot '
    'distinguish a usable load profile from a flat line through the middle of one.'
).format(split=SPLIT)

if not ACTIVE_REALISM:
    print('No realism metrics in this run — nothing to emit.')
else:
    tex_agg = to_latex_paper(
        realism_median,
        caption='Curve realism for individual energy-profile prediction.',
        label=AGG_LABEL,
        note=REALISM_NOTE)
    print('% ===== REALISM, ALL PROCESSES ====='); print(tex_agg)

    if SAVE_LATEX:
        Path('visuals').mkdir(exist_ok=True)
        _out = Path('visuals') / 'individual_profile_realism.tex'
        _out.write_text('% Aggregated realism table — \\input{} this file.\n' + tex_agg + '\n')
        print(f'\nSaved: {_out}')


% ===== REALISM, ALL PROCESSES =====
\begin{table*}[H]
\centering

\begin{minipage}{8.5cm}
\centering

\captionsetup{
    justification=centering,
    singlelinecheck=false,
    format=plain
}

\caption{Curve realism for individual energy-profile prediction.}
\label{tab:individual_profile_realism}

\vspace{-0.5em}

\begin{tabular}{p{6cm}|c|c|c|c|c|c}
\toprule
\textbf{Method} & \textbf{Sum} & \textbf{Max} & \textbf{Mean} & \textbf{Std} & \textbf{Roughness} & \textbf{Overall} \\
\midrule
Step DTW smooth + ML + Ext. & 0.018 & \textbf{0.070} & 0.048 & \textbf{0.467} & \textbf{0.569} & \textbf{0.234} \\
Median per activity and sensor & \textbf{0.016} & 0.072 & \textbf{0.043} & 0.552 & 0.662 & 0.269 \\
DTW + ML + Ext. Factors & 0.018 & 0.080 & 0.050 & 0.531 & 0.676 & 0.271 \\
ML only (no DTW) & 0.018 & 0.089 & 0.054 & 0.592 & 0.702 & 0.291 \\
Baseline & 0.018 & 0.074 & 0.051 & 0.613 & 0.724 & 0.296 \\
\bottomrule
\end{tabular}

\vspace{0.5em}

\parbox{12cm}{%
\footnotesize
Curve realism for 

## 3 · Breakdown — what the aggregated table is made of

Every number in section 1 is a median of medians, and this section shows the levels
underneath it, tables only:

1. **Unit** — one (process, activity, sensor) cell, the individual unit that later
   gets aggregated. This is the population of the aggregated table: one cell, one
   vote, so `cells` is 1 on every row.
2. **Activity** — the same cells collapsed by activity.
3. **Sensor** — the same cells collapsed by sensor.

`curves` is how many per-curve evaluations sit behind the row and `cells` how many
(process, activity, sensor) units — so a row backed by one cell and three curves can
be read for what it is.

There is no separate *object* dimension in `curve_eval_results.parquet`: the identity
columns are Process, Activity, Sensor and Instance. The object/plant level **is**
`Process`, and it appears only as part of the unit key below — the per-process result
tables were removed on request.

All three identity columns are **anonymised**: the labels are shuffled numbers, so
they group the rows correctly but name nothing.

### Reading the zeros

Two different things print as `0.000`, and only one of them is rounding:

- **Rounded to three decimals.** `Sum` and `Mean` are normalised by the sensor's
  typical real magnitude, and for a well-predicted cell the residual is genuinely
  ~1e-3 — the median `Sum` across all units is 0.018. About 8% of `Sum` rows round
  to `0.000` without being zero.
- **Exactly zero — an idle cell.** 63 of the 1,950 unit rows are 0 in *every*
  column at once. Those are the 8 (process, activity, sensor) cells whose **real
  curve is identically zero**: standstill, fault and cleaning states where the
  sensor reads 0 for the whole activity. The prediction is also 0, so the error is exactly 0 and every
  method scores a free perfect cell. They sit in `Process 6` (6 cells) and
  `Process 2` (2 cells) and they flatter every method equally, so they do not
  change the ranking — but they do pull all five medians slightly toward 0.


In [7]:
# Median realism at each level of the ladder, with the population behind every row.
# All three tables aggregate the SAME per-curve values with the SAME statistic as
# section 1 — only the grouping widens, so the last one reproduces the aggregated
# table exactly when grouped by Approach alone.
# The unit is keyed on Process too: the same activity/sensor name occurs in more
# than one process ('Feed vorwärts' does), and without Process those two cells
# would silently merge into one row — exactly the aggregation this level exists to
# undo. Process is an identifier here, not a result: there are no per-process
# tables in this notebook.
LADDER = [('Unit — one (process, activity, sensor) cell', ['Process', 'Activity', 'Sensor']),
          ('Activity',                                    ['Activity']),
          ('Sensor',                                      ['Sensor'])]

def breakdown(keys):
    """Median over the cells of each `keys` group, per approach, plus the counts."""
    grp = keys + ['Approach']
    t = combo.groupby(grp)[ACTIVE_REALISM].median()
    t['Overall'] = t.mean(axis=1)                       # same definition as section 1
    t['cells']   = combo.groupby(grp).size()            # (process, activity, sensor) units
    t['curves']  = raw.groupby(grp).size()              # per-curve evaluations behind them
    return t.sort_values(keys + ['Overall'])

if not ACTIVE_REALISM:
    display(Markdown('> **Skipped — no realism metrics in this run.**'))
else:
    # Rendered as bare HTML rather than through display(df) or a Styler: at ~2,000
    # rows both alternatives roughly double the size of the .ipynb — display(df)
    # stores a text/plain copy of every row alongside the HTML, and a Styler emits
    # per-cell markup. There is no single "best" row to highlight inside a
    # breakdown anyway.
    from IPython.display import HTML
    for title, keys in LADDER:
        t = breakdown(keys).round(3)
        display(Markdown(f'### {title} — {t.index.droplevel(-1).nunique()} groups, '
                         f'{len(t):,} rows ({t["curves"].sum():,} curves)'))
        display(HTML(t.to_html()))


### Unit — one (process, activity, sensor) cell — 390 groups, 1,950 rows (96,725 curves)

### Activity — 26 groups, 130 rows (96,725 curves)

### Sensor — 57 groups, 285 rows (96,725 curves)